# Hardread TCG — Baseline Agent

Kaggle runner for the Pokémon TCG AI Battle Challenge.
Defines the `agent` function for submission and runs evaluation self-play.

In [ ]:
import os
import random
import time
import json
from pathlib import Path
from typing import Dict, List
from kaggle_environments import make

## Deck

Read the sample deck. In the competition, `agent()` returns this deck during the initial selection phase.

In [ ]:
# Hardcoded sample deck fallback (uses official sample_submission/deck.csv IDs).
SAMPLE_DECK = [1158, 721, 721, 722, 722, 722, 722, 723, 723, 723, 723, 1145, 1145, 1145, 1145, 1205, 1205, 1227, 1227, 1227, 1227, 1235, 1235, 1235, 1235] + [3] * 35

def read_deck_csv() -> List[int]:
    """Read deck.csv from competition data or local path."""
    candidates = [
        "deck.csv",
        "/kaggle_simulations/agent/deck.csv",
        "/kaggle/input/pokemon-tcg-ai-battle/sample_submission/deck.csv",
    ]
    for path in candidates:
        if os.path.exists(path):
            with open(path, "r") as f:
                return [int(line.strip()) for line in f if line.strip()]
    return SAMPLE_DECK

DECK = read_deck_csv()
print(f"Loaded deck with {len(DECK)} cards")

## Agent — ε-greedy contextual bandit (v1)

Self-contained bandit agent. Learns within a single game from a shaped per-step
reward (change in prize differential + small HP differential). Arms are keyed by
(select.type, option.type). On the initial selection (select is None) it returns the deck.

In [ ]:
import random as _rng
from typing import Any, Dict, List, Tuple


def _safe_int(value: Any, default: int = 0) -> int:
    try:
        return int(value)
    except (TypeError, ValueError):
        return default


def _prizes_left(player: Any) -> int:
    if not isinstance(player, dict):
        return 6
    return sum(1 for p in (player.get("prize") or []) if p is not None)


def _active_hp(player: Any) -> int:
    if not isinstance(player, dict):
        return 0
    active = player.get("active") or []
    if not active or not isinstance(active[0], dict):
        return 0
    return _safe_int(active[0].get("hp"), 0)


class BanditAgent:
    def __init__(self, deck: List[int], alpha: float = 0.1, epsilon: float = 0.1,
                 hp_weight: float = 0.01, seed: int = None):
        self.deck = list(deck)
        self.alpha = alpha
        self.epsilon = epsilon
        self.hp_weight = hp_weight
        self.q: Dict[Tuple[int, int], float] = {}
        self.n: Dict[Tuple[int, int], int] = {}
        self.last_arm = None
        self.last_signal = None
        self.rng = _rng.Random(seed)

    def _signal(self, obs: Dict) -> Tuple[int, int]:
        cur = obs.get("current") if isinstance(obs, dict) else None
        if not isinstance(cur, dict):
            return (6, 0)
        players = cur.get("players") or []
        if len(players) < 2:
            return (6, 0)
        return (_prizes_left(players[1]) - _prizes_left(players[0]),
                _active_hp(players[1]) - _active_hp(players[0]))

    def _observe(self, obs: Dict) -> None:
        if self.last_arm is None or self.last_signal is None:
            return
        sig = self._signal(obs)
        reward = (sig[0] - self.last_signal[0]) + self.hp_weight * (sig[1] - self.last_signal[1])
        self.q[self.last_arm] = self.q.get(self.last_arm, 0.0) + self.alpha * (reward - self.q.get(self.last_arm, 0.0))
        self.n[self.last_arm] = self.n.get(self.last_arm, 0) + 1
        self.last_arm = None
        self.last_signal = None

    def _arm(self, select: Dict, option: Any) -> Tuple[int, int]:
        st = _safe_int(select.get("type"), -1)
        ot = _safe_int(option.get("type"), -1) if isinstance(option, dict) else -1
        return (st, ot)

    def act(self, obs: Any) -> List[int]:
        if obs is None or isinstance(obs, list) or (isinstance(obs, dict) and obs.get("select") is None):
            return list(self.deck)
        self._observe(obs)
        select = obs["select"]
        options = select["option"]
        min_c = _safe_int(select.get("minCount", select.get("maxCount", 1)), 1)
        max_c = _safe_int(select.get("maxCount", min_c), min_c)
        count = max(min_c, min(max_c, len(options)))
        if count <= 0:
            return []
        arms = [self._arm(select, options[i]) for i in range(len(options))]
        if self.rng.random() < self.epsilon:
            order = list(range(len(options)))
            self.rng.shuffle(order)
            chosen = order[:count]
        else:
            chosen = sorted(range(len(options)), key=lambda i: (self.q.get(arms[i], 0.0), -i), reverse=True)[:count]
        self.last_arm = arms[chosen[0]]
        self.last_signal = self._signal(obs)
        return chosen

    def stats(self) -> Dict[str, Any]:
        return {"n_arms": len(self.q), "n_updates": sum(self.n.values()), "epsilon": self.epsilon}


_BANDIT = BanditAgent(DECK, seed=42)

def agent(obs, *args) -> List[int]:
    return _BANDIT.act(obs)

print(f"Bandit agent ready: {_BANDIT.stats()}, deck={len(DECK)} cards")


## Self-Play Evaluation

Run N games of random-vs-random to verify the agent works.

In [ ]:
N_GAMES = 10

# Inspect cg.api module for type enums and constants
try:
    import cg.api as cgapi
    print("=== cg.api contents ===")
    print(f"dir(cg.api): {[x for x in dir(cgapi) if not x.startswith('_')]}")
    for name in dir(cgapi):
        obj = getattr(cgapi, name)
        if isinstance(obj, type) and not name.startswith('_'):
            print(f"\nClass {name}:")
            for attr in dir(obj):
                if not attr.startswith('_'):
                    val = getattr(obj, attr)
                    if not callable(val):
                        print(f"  {attr} = {val!r}")
except Exception as e:
    print(f"cg.api import failed: {e}")
print("=== end cg.api ===\n")


# Two independent bandit instances for self-play so the players don't share Q-tables.
def make_bandit_agent(seed: int):
    b = BanditAgent(DECK, seed=seed)
    def fn(obs, *args):
        return b.act(obs)
    fn._bandit = b
    return fn


def dump_observation(obs, label="obs"):
    if obs is None:
        print(f"{label}: None")
        return
    print(f"{label} type: {type(obs).__name__}")
    if isinstance(obs, dict):
        print(f"  keys: {list(obs.keys())}")
        for k, v in obs.items():
            if k == "select" and isinstance(v, dict):
                print(f"    select keys: {list(v.keys())}")
                print(f"      option: list[{len(v.get('option', []))}]")
                opts = v.get('option', [])
                if opts:
                    print(f"      option[0] type: {type(opts[0]).__name__}")
                    if isinstance(opts[0], dict):
                        print(f"      option[0] keys: {list(opts[0].keys())}")
                        for ok, ov in opts[0].items():
                            if isinstance(ov, list):
                                print(f"        {ok}: list[{len(ov)}]")
                            elif isinstance(ov, dict):
                                print(f"        {ok}: dict keys={list(ov.keys())}")
                            else:
                                print(f"        {ok}: {ov!r}")
                print(f"      maxCount: {v.get('maxCount')}")
                print(f"      minCount: {v.get('minCount')}")
            elif k == "observation" and isinstance(v, dict):
                print(f"    observation keys: {list(v.keys())}")
                for ok, ov in v.items():
                    if isinstance(ov, list):
                        print(f"      {ok}: list[{len(ov)}]")
                    elif isinstance(ov, dict):
                        print(f"      {ok}: dict keys={list(ov.keys())}")
                    else:
                        print(f"      {ok}: {ov!r}")
            elif isinstance(v, list):
                print(f"    {k}: list[{len(v)}]")
            elif isinstance(v, dict):
                print(f"    {k}: dict keys={list(v.keys())}")
            else:
                print(f"    {k}: {v!r}")


def extract_winner(trajectory: List) -> int:
    if not trajectory:
        return -1
    last = trajectory[-1]
    if isinstance(last, list) and len(last) >= 2:
        r0 = last[0].get("reward") if isinstance(last[0], dict) else None
        r1 = last[1].get("reward") if isinstance(last[1], dict) else None
        if r0 is not None and r1 is not None:
            if r0 > r1:
                return 0
            elif r1 > r0:
                return 1
    return -1


results = []
wins = 0
replay_traj = None
for i in range(N_GAMES):
    p0 = make_bandit_agent(seed=42 + i)
    p1 = make_bandit_agent(seed=100 + i)
    env = make("cabt", configuration={"decks": [DECK, DECK]})
    t0 = time.time()
    trajectory = env.run([p0, p1])
    elapsed = time.time() - t0
    steps = len(trajectory)
    if i == 0:
        replay_traj = trajectory

    if i == 0 and trajectory:
        print(f"\n=== GAME 0 TRAJECTORY DUMP ===")
        print(f"Trajectory length: {len(trajectory)}")
        print(f"Entry 0 type: {type(trajectory[0]).__name__}")
        if isinstance(trajectory[0], list):
            print(f"  entry[0][0] keys: {list(trajectory[0][0].keys())}")
            print(f"  entry[0][1] keys: {list(trajectory[0][1].keys())}")
            dump_observation(trajectory[0][0].get("observation"), "P0 obs step 0")
            dump_observation(trajectory[0][1].get("observation"), "P1 obs step 0")
        mid = len(trajectory) // 2
        if mid < len(trajectory) and isinstance(trajectory[mid], list):
            dump_observation(trajectory[mid][0].get("observation"), f"P0 obs step {mid}")
        print("=== END DUMP ===\n")

    winner = extract_winner(trajectory)
    if winner == 0:
        wins += 1
    results.append({"game": i, "steps": steps, "time": round(elapsed, 2), "winner": winner,
                    "p0_stats": p0._bandit.stats(), "p1_stats": p1._bandit.stats()})
    print(f"Game {i+1}/{N_GAMES}: {steps} steps, {elapsed:.1f}s, winner=P{winner if winner >=0 else '?'}, p0={p0._bandit.stats()}")

avg_steps = sum(r["steps"] for r in results) / len(results)
avg_time = sum(r["time"] for r in results) / len(results)
win_rate = wins / N_GAMES
print(f"\nDone: {N_GAMES} games, avg {avg_steps:.0f} steps, avg {avg_time:.1f}s, P0 win rate: {win_rate:.0%}")


In [ ]:
out_dir = Path("/kaggle/working")
out_dir.mkdir(parents=True, exist_ok=True)
with open(out_dir / "results.json", "w") as f:
    json.dump({
        "n_games": N_GAMES,
        "agent": "bandit-v1",
        "avg_steps": round(avg_steps, 1),
        "avg_time": round(avg_time, 1),
        "win_rate": round(win_rate, 3),
        "results": results,
    }, f, indent=2)
print("Results saved to /kaggle/working/results.json")


## Text Replay of Game 0

Renders the first game's trajectory as a human-readable play-by-play and writes it to `replay.txt`.

In [ ]:
SELECT_TYPE_NAMES = {
    0: "NONE", 1: "MAIN", 2: "CARD", 3: "ATTACHED_CARD", 4: "CARD_OR_ATTACHED",
    5: "ENERGY", 6: "SKILL", 7: "ATTACK", 8: "EVOLVE", 9: "COUNT", 10: "YES_NO",
    11: "SPECIAL_CONDITION",
}
SELECT_CONTEXT_NAMES = {
    1: "MAIN", 2: "SETUP_ACTIVE", 3: "SETUP_BENCH", 4: "SWITCH", 5: "TO_ACTIVE",
    6: "TO_BENCH", 36: "ATTACK", 38: "EVOLVE", 42: "IS_FIRST", 43: "MULLIGAN", 44: "ACTIVATE",
}
LOG_TYPE_NAMES = {
    1: "MULLIGAN_CHECK", 2: "DRAW", 3: "PASS", 4: "DRAW_CARD", 5: "TURN_START",
    6: "MOVE_CARD", 7: "PLAY_CARD", 11: "ATTACH_ENERGY", 12: "ATTACK",
}
AREA_NAMES = {
    1: "deck", 2: "hand", 3: "active", 4: "discard", 5: "bench", 6: "stadium",
    7: "prize", 8: "attached", 9: "lost_zone",
}

_CARD_NAMES = {}
for _csv_path in ["/kaggle/input/pokemon-tcg-ai-battle/EN Card Data.csv", "EN Card Data.csv", "Data/EN_Card_Data.csv"]:
    try:
        import csv as _csv
        with open(_csv_path) as _f:
            for _r in _csv.DictReader(_f):
                _CARD_NAMES[int(_r["Card ID"])] = _r["Card Name"].strip()
        break
    except Exception:
        continue
if not _CARD_NAMES:
    _CARD_NAMES = {3: "Water Energy", 721: "Kyogre", 722: "Snover", 723: "Mega Abomasnow ex",
                   1158: "Maximum Belt", 1145: "Mega Signal", 1205: "Cyrano",
                   1227: "Lillie's Determination", 1235: "Waitress"}

def _cname(cid):
    try: cid = int(cid)
    except (TypeError, ValueError): return str(cid)
    return _CARD_NAMES.get(cid, f"#{cid}")

def _opt_desc(option):
    if not isinstance(option, dict):
        return str(option)
    ot = SELECT_TYPE_NAMES.get(_si(option.get("type"), -1), str(option.get("type")))
    cid = option.get("cardId") or option.get("id")
    name = _cname(cid) if cid is not None else None
    if name:
        return f"{name} ({ot})"
    return ot

def _action_desc(action, options):
    if not isinstance(action, list):
        return str(action)
    if not action:
        return "[]"
    if isinstance(action[0], int) and len(action) > 5:
        from collections import Counter
        counts = Counter(action)
        return ", ".join(f"{_cname(c)} x{n}" if n > 1 else _cname(c) for c, n in sorted(counts.items(), key=lambda x: str(x[0])))
    parts = []
    for idx in action:
        try: idx = int(idx)
        except (TypeError, ValueError): parts.append(str(idx)); continue
        if isinstance(options, list) and 0 <= idx < len(options):
            parts.append(f"[{idx}] {_opt_desc(options[idx])}")
        else:
            parts.append(f"[{idx}]")
    return ", ".join(parts)

def _si(v, d=0):
    try: return int(v)
    except (TypeError, ValueError): return d

def _prizes_left(p):
    if not isinstance(p, dict): return 6
    return sum(1 for x in (p.get("prize") or []) if x is not None)

def _active_info(p):
    if not isinstance(p, dict): return ("-", 0)
    act = p.get("active") or []
    if not act or not isinstance(act[0], dict): return ("-", 0)
    a = act[0]
    name = a.get("name") or a.get("cardName") or _cname(a.get("id") or a.get("cardId") or "?")
    return (str(name), _si(a.get("hp"), 0))

def _status_flags(p):
    if not isinstance(p, dict): return ""
    flags = []
    for f in ("poisoned", "burned", "asleep", "paralyzed", "confused"):
        if p.get(f): flags.append(f[:3].upper())
    return ",".join(flags)

def _board_line(label, p):
    name, hp = _active_info(p)
    prizes = _prizes_left(p)
    hand = _si(p.get("handCount"), 0) if isinstance(p, dict) else 0
    bench = len(p.get("bench") or []) if isinstance(p, dict) else 0
    deck = _si(p.get("deckCount"), 0) if isinstance(p, dict) else 0
    st = _status_flags(p)
    return f"  {label}: active={name} HP={hp} prizes={prizes} hand={hand} bench={bench} deck={deck} {st}"

def render_replay(traj):
    if not traj:
        return "(empty trajectory)"
    lines = []
    lines.append(f"=== GAME 0 REPLAY ({len(traj)} steps) ===")
    for step, entry in enumerate(traj):
        if not isinstance(entry, list) or len(entry) < 2:
            lines.append(f"\n[step {step}] (malformed)")
            continue
        s0, s1 = entry[0], entry[1]
        obs0 = s0.get("observation") if isinstance(s0, dict) else None
        obs1 = s1.get("observation") if isinstance(s1, dict) else None
        cur = obs0.get("current") if isinstance(obs0, dict) else None
        turn = _si(cur.get("turn"), 0) if isinstance(cur, dict) else 0
        players = cur.get("players") if isinstance(cur, dict) else None
        lines.append(f"\n[step {step}] turn={turn}")
        if isinstance(players, list) and len(players) >= 2:
            lines.append(_board_line("P0", players[0]))
            lines.append(_board_line("P1", players[1]))
        for who, s, obs in (("P0", s0, obs0), ("P1", s1, obs1)):
            sel = obs.get("select") if isinstance(obs, dict) else None
            if isinstance(sel, dict):
                stype = SELECT_TYPE_NAMES.get(_si(sel.get("type"), -1), str(sel.get("type")))
                sctx = SELECT_CONTEXT_NAMES.get(_si(sel.get("context"), -1), str(sel.get("context")))
                nopt = len(sel.get("option") or [])
                mc = sel.get("maxCount")
                action = s.get("action") if isinstance(s, dict) else None
                opts = sel.get("option") or []
                lines.append(f"  {who} chooses: type={stype} ctx={sctx} options={nopt} maxCount={mc} -> {_action_desc(action, opts)}")
        logs = obs0.get("logs") if isinstance(obs0, dict) else None
        if isinstance(logs, list) and logs:
            tail = logs[-3:]
            for lg in tail:
                if isinstance(lg, dict):
                    ltype = LOG_TYPE_NAMES.get(_si(lg.get("type"), -1), str(lg.get("type")))
                    parts = [f"{ltype}", f"P{_si(lg.get('playerIndex'), -1)}"]
                    for k, v in lg.items():
                        if k in ("type", "playerIndex"):
                            continue
                        if k in ("cardId", "cardIdTarget") and v is not None:
                            parts.append(f"{k}={_cname(v)}")
                        elif k in ("fromArea", "toArea") and v is not None:
                            parts.append(f"{k}={AREA_NAMES.get(_si(v), str(v))}")
                        else:
                            parts.append(f"{k}={v}")
                    txt = " ".join(parts)
                else:
                    txt = str(lg).strip()
                if txt:
                    lines.append(f"  log: {txt[:200]}")
    lines.append("\n=== END REPLAY ===")
    return "\n".join(lines)

replay_text = render_replay(replay_traj)
with open(out_dir / "replay.txt", "w") as f:
    f.write(replay_text)
print("Replay saved to /kaggle/working/replay.txt")
print("\n".join(replay_text.splitlines()[:60]))
